# 🌸 Langfuse — Complete Instructor Guide
### Open-Source LLM Observability · Tracing · Prompt Management · Evaluation · Experiments
---
> **Models used:** `gpt-4o-mini` (chat) · `text-embedding-3-small` (embeddings)  
> **Packages:** `langfuse>=4.6.0` (v4, OTEL-based) · `langchain==1.3.x` · `langchain-openai==1.2.x`  
> **Secrets loaded from:** Google Colab Secret Manager (🔑 left panel)

---

## 📚 Table of Contents
1. [Detailed Notes — What Is Langfuse?](#notes)
2. [Setup & Installation](#setup)
3. [Configure Secrets (Colab)](#secrets)
4. [Module 1 — Core Tracing with @observe Decorator](#observe)
5. [Module 2 — Manual Tracing (Spans, Generations, Events)](#manual)
6. [Module 3 — LangChain Integration with CallbackHandler](#langchain)
7. [Module 4 — Prompt Management (Create, Version, Pull)](#prompts)
8. [Module 5 — Scoring & Evaluation](#scoring)
9. [Module 6 — Datasets & Experiments](#datasets)
10. [Module 7 — Full Production RAG Pipeline](#rag)
11. [Module 8 — Langfuse vs LangSmith — Side-by-Side Comparison](#comparison)


---
## 1. 📖 Detailed Notes — What Is Langfuse? <a name="notes"></a>

### 1.1 Background & Philosophy

**Langfuse** (founded 2023, open-source since launch) is an LLM observability and management
platform that takes a framework-agnostic approach. While LangSmith is deeply integrated with
LangChain, Langfuse is designed to work with **any** LLM stack through multiple integration
options.

Key design philosophy:
- **Open-source first**: The entire platform is available as open-source (MIT license)
- **Self-hostable**: Run on your own infrastructure for full data privacy
- **Framework-agnostic**: Works with OpenAI SDK, LangChain, LlamaIndex, Haystack, raw HTTP
- **Developer-first**: Designed for the full dev workflow, not just production monitoring

---

### 1.2 Core Architecture

```
Your Application
       │
       ▼
┌─────────────────────────────────────────────────────────┐
│                    Langfuse SDK / CallbackHandler        │
│   @observe decorator  │  Manual SDK calls  │  OTEL      │
└─────────────────────────────────────────────────────────┘
       │  async batch export (non-blocking)
       ▼
┌─────────────────────────────────────────────────────────┐
│              Langfuse Server (cloud or self-hosted)      │
│   ┌──────────┐  ┌──────────┐  ┌──────────┐  ┌───────┐  │
│   │  Traces  │  │ Prompts  │  │ Datasets │  │Scores │  │
│   └──────────┘  └──────────┘  └──────────┘  └───────┘  │
└─────────────────────────────────────────────────────────┘
       │
       ▼
┌─────────────────────────────────────────────────────────┐
│              Langfuse UI (langfuse.com or localhost)     │
│   Trace viewer │ Prompt editor │ Experiments │ Analytics │
└─────────────────────────────────────────────────────────┘
```

---

### 1.3 Data Model — Key Concepts

| Concept | Definition | Analogy |
|---------|-----------|---------|
| **Trace** | Root object for one user interaction/request | A "session" or "transaction" |
| **Span** | A unit of work within a trace (retrieval, tool call) | A function call |
| **Generation** | A special span for LLM/model API calls | The actual "thinking" step |
| **Event** | An instant timestamp (log entry, user action) | A log.info() statement |
| **Score** | A numeric or categorical quality rating attached to a trace or span | An eval result |
| **Session** | Groups multiple traces from one user session | A chat conversation |
| **User** | User ID attached to traces for per-user analytics | A customer identifier |

**Hierarchy:**
```
Trace (root)
  └── Span: "retrieval"
  │     └── Event: "cache_miss"
  └── Span: "prompt-building"
  └── Generation: "llm-call"  ← contains: model, prompt, completion, tokens, cost
        └── Span: "post-processing"
```

---

### 1.4 Three Ways to Instrument with Langfuse

#### Method 1: @observe Decorator (Recommended for new code)
```python
from langfuse import observe, get_client

@observe()                    # Creates a span automatically
def my_function(input: str) -> str:
    return process(input)

@observe(as_type="generation")  # Creates a generation span
def call_llm(prompt: str) -> str:
    return llm.invoke(prompt)
```

#### Method 2: CallbackHandler for LangChain (Drop-in for existing LangChain code)
```python
from langfuse import observe, get_client, propagate_attributes
from langfuse.langchain import CallbackHandler

@observe(name="my-pipeline")
def run(inputs):
    lf = get_client()
    with propagate_attributes(user_id="u-1", session_id="s-1", tags=["prod"]):
        handler = CallbackHandler(trace_context={"trace_id": lf.get_current_trace_id()})
        return chain.invoke(inputs, config={"callbacks": [handler]})
```

#### Method 3: Manual SDK / OTEL spans (Maximum control)
```python
# v4: trace() / span() / generation() removed — use start_as_current_observation()
from langfuse import Langfuse, propagate_attributes

langfuse = Langfuse()
with langfuse.start_as_current_observation(name="my-pipeline", as_type="span") as root:
    with propagate_attributes(user_id="user-123", session_id="sess-1"):
        retrieval = root.start_observation(name="retrieval", as_type="span")
        retrieval.end()
        gen = root.start_observation(name="llm-call", as_type="generation", model="gpt-4o-mini")
        gen.update(output="response text")
        gen.end()
langfuse.flush()
```

---

### 1.5 Langfuse Prompt Management

Langfuse has a built-in prompt management system with:
- **Versioned prompts**: Every prompt change creates a new version
- **Labels**: Tag versions as `production`, `staging`, `latest`
- **Config variables**: Prompts are templates with `{{variable}}` placeholders
- **Model config**: Attach model name, temperature, etc. to a prompt version
- **SDK integration**: Pull prompts at runtime — changes take effect without redeployment

**Prompt lifecycle:**
```
Create in Langfuse UI or SDK
       ↓
Version 1 (label: latest)
       ↓
Update → Version 2 (label: latest)
       ↓  
Test → promote Version 2 (label: production)
       ↓
All running apps pull "production" → automatically get Version 2
```

---

### 1.6 Langfuse vs LangSmith — Key Differences

| Aspect | Langfuse | LangSmith |
|--------|---------|-----------|
| **Open-source** | ✅ Full platform (MIT) | ❌ Closed-source |
| **Self-hostable** | ✅ Docker, K8s, or managed | ❌ Cloud only |
| **Framework support** | Any framework, OTEL | Primarily LangChain |
| **LangChain integration** | Good (CallbackHandler) | Native/best-in-class |
| **Prompt management** | ✅ Built-in, very good | ✅ Hub, very good |
| **Datasets/Evals** | ✅ Built-in | ✅ Built-in |
| **Cost tracking** | ✅ Per model pricing built-in | ✅ Yes |
| **UI quality** | ⭐⭐⭐⭐ | ⭐⭐⭐⭐⭐ |
| **Free tier** | Generous (100k observations/mo) | Limited |
| **Data privacy** | Full (self-host) | Data goes to LangChain servers |
| **Best for** | Privacy-sensitive, non-LangChain, open-source fans | LangChain-heavy teams |

---

### 1.7 When to Choose Langfuse Over LangSmith

Choose **Langfuse** when:
- You have **data privacy requirements** (HIPAA, GDPR, on-prem) — self-host it
- You're **not using LangChain** (e.g., raw OpenAI SDK, LlamaIndex, custom)
- You want an **open-source** tool you can modify and extend
- You need **multi-framework** support in one platform
- Your team prefers **per-observation pricing** vs per-seat LangSmith pricing

Choose **LangSmith** when:
- Your team is **all-in on LangChain** and wants the tightest integration
- You need the **most polished UI** for developer experience
- You want **LangChain Hub** for sharing prompts with the community
- You're already paying for LangChain's enterprise tier


---
## 2. 🛠️ Setup & Installation <a name="setup"></a>


In [ ]:
# ── Install all dependencies ──────────────────────────────────────────────────
!pip install -q \
    langfuse>=4.6.0 \
    langchain>=1.3.0 \
    langchain-openai>=1.2.0 \
    langchain-core>=0.3.0 \
    langchain-text-splitters>=0.3.0 \
    langchain-community>=0.4.0 \
    langchain-chroma>=0.2.0 \
    chromadb \
    openai \
    tiktoken

print("✅ All packages installed")
print()
import langfuse, langchain, langchain_openai
print(f"  langfuse:        {langfuse.__version__}")
print(f"  langchain:       {langchain.__version__}")


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.
google-adk 1.29.0 requires opentelemetry-api<1.39.0,>=1.36.0, but you have opentelemetry-api 1.42.1 which is incompatible.
google-adk 1.29.0 requires opentelemetry-sdk<1.39.0,>=1.36.0, but you have opentelemetry-sdk 1.42.1 which is incompatible.
opentelemetry-exporter-gcp-logging 1.11.0a0 requires opentelemetry-sdk<1.39.0,>=1.35.0, but you have opentelemetry-sdk 1.42.1 which is incompatible.
✅ All packages installed

  langfuse:        4.6.1
  langchain:       1.2.15


---
## 3. 🔑 Configure Secrets (Colab Secret Manager) <a name="secrets"></a>

**Before running this cell, add these secrets in the 🔑 left panel:**

| Secret Name | Where to Find |
|------------|---------------|
| `OPENAI_API_KEY` | platform.openai.com → API Keys |
| `LANGFUSE_PUBLIC_KEY` | cloud.langfuse.com → Settings → API Keys → Public Key |
| `LANGFUSE_SECRET_KEY` | cloud.langfuse.com → Settings → API Keys → Secret Key |
| `LANGFUSE_HOST` | `https://cloud.langfuse.com` (or your self-hosted URL) |

> **Self-hosting:** If running Langfuse locally with Docker, set `LANGFUSE_HOST=http://localhost:3000`


In [ ]:
import os
from google.colab import userdata

# ── Load secrets from Colab Secret Manager ────────────────────────────────────
os.environ["OPENAI_API_KEY"]        = userdata.get("OPENAI_API_KEY")
os.environ["LANGFUSE_PUBLIC_KEY"]   = userdata.get("LANGFUSE_PUBLIC_KEY")
os.environ["LANGFUSE_SECRET_KEY"]   = userdata.get("LANGFUSE_SECRET_KEY")
os.environ["LANGFUSE_HOST"]         = userdata.get("LANGFUSE_HOST") or "https://cloud.langfuse.com"

# ── Initialise Langfuse client and verify connection ─────────────────────────
from langfuse import Langfuse

langfuse_client = Langfuse(
    public_key  = os.environ["LANGFUSE_PUBLIC_KEY"],
    secret_key  = os.environ["LANGFUSE_SECRET_KEY"],
    host        = os.environ["LANGFUSE_HOST"],   # ← v4: use host= (base_url= also accepted)
    debug       = False,
    flush_at    = 50,
    flush_interval = 5.0,
)

# Verify connection
try:
    auth_ok = langfuse_client.auth_check()
    print(f"✅ Connected to Langfuse at {os.environ['LANGFUSE_HOST']}")
    print(f"   Auth check: {'OK' if auth_ok else 'Failed'}")
except Exception as e:
    print(f"❌ Connection failed: {e}")
    print("   Check your API keys and host URL")


✅ Connected to Langfuse at https://us.cloud.langfuse.com
   Auth check: OK


---
## 4. 🎯 Module 1 — Core Tracing with @observe Decorator <a name="observe"></a>

The `@observe` decorator is the **recommended way** to instrument code in Langfuse 4.x.
It automatically:
- Creates a trace (for the outermost function) or span (for nested functions)
- Captures inputs and outputs
- Measures latency
- Links parent-child relationships in the trace tree


In [ ]:
# v4 change: propagate_attributes() is the new way to attach user_id, session_id, tags
from langfuse import observe, get_client, propagate_attributes
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# Models
llm        = ChatOpenAI(model="gpt-4o-mini", temperature=0.3)
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

# ── Basic @observe usage ───────────────────────────────────────────────────────

@observe()   # Creates a TRACE when root, SPAN when nested
def step_1_preprocess(user_query: str) -> str:
    """Cleans and normalises the user query."""
    return user_query.strip().lower()


@observe()
def step_2_retrieve(query: str) -> list[str]:
    """Simulates document retrieval."""
    mock_docs = [
        f"Document about {query}: LLMOps involves managing LLM systems in production.",
        f"Reference: Key tools include MLflow, LangSmith, Langfuse, vLLM."
    ]
    return mock_docs


@observe(as_type="generation")  # Tag as generation span (shows token counts in UI)
def step_3_generate(query: str, context: list[str]) -> str:
    """Calls the LLM to generate an answer."""
    context_str = "\n".join(context)
    prompt = ChatPromptTemplate.from_messages([
        ("system", f"Answer based on this context:\n{context_str}"),
        ("human", "{query}")
    ])
    chain  = prompt | llm | StrOutputParser()
    return chain.invoke({"query": query})


@observe(name="full-qa-pipeline")  # Root trace
def run_pipeline(user_input: str) -> dict:
    """
    Full pipeline — each step becomes a child span under this root trace.
    Langfuse v4: use propagate_attributes() to attach user_id, session_id, tags.
    """
    lf = get_client()

    # v4: propagate_attributes() sets user_id/session_id/tags on the current span
    # and ALL child spans created within this context manager.
    with propagate_attributes(
        user_id="student-demo-001",
        session_id="colab-demo-session",
        tags=["demo", "day-14", "langfuse-tutorial"],
    ):
        # Metadata still set via update_current_span (metadata= accepts any value)
        lf.update_current_span(metadata={"lesson": "Module 1", "topic": "observe-decorator"})

        clean_query = step_1_preprocess(user_input)
        docs        = step_2_retrieve(clean_query)
        answer      = step_3_generate(clean_query, docs)

    return {"query": user_input, "answer": answer, "num_docs": len(docs)}


# ── Run the pipeline ───────────────────────────────────────────────────────────
result = run_pipeline("What tools are used in LLMOps?")
print("Answer:", result["answer"])
print()
print("✅ Trace created in Langfuse!")
print(f"   View at: {os.environ.get('LANGFUSE_HOST', 'https://cloud.langfuse.com')}")
print("   Trace tree: full-qa-pipeline → step_1 → step_2 → step_3")


Answer: LLMOps involves managing LLM systems in production, and key tools used in this field include MLflow, LangSmith, Langfuse, and vLLM.

✅ Trace created in Langfuse!
   View at: https://us.cloud.langfuse.com
   Trace tree: full-qa-pipeline → step_1 → step_2 → step_3


In [ ]:
# ── @observe with custom input/output capture ─────────────────────────────────
# v4 change: update_current_observation() removed → use update_current_span()
#            or update_current_generation() depending on as_type

@observe(
    name="structured-generation",
    as_type="generation",
    capture_input=True,    # Default: True — set False for sensitive data
    capture_output=True,   # Default: True — set False for PII outputs
)
def generate_structured_answer(
    question: str,
    context: str,
    user_id: str
) -> str:
    """
    Advanced @observe usage with v4 update_current_generation().
    """
    lf = get_client()

    # v4: update_current_generation() (was update_current_observation() in v2/v3)
    lf.update_current_generation(
        metadata={
            "user_id": user_id,
            "model": "gpt-4o-mini",
            "context_length": len(context.split()),
        },
        level="DEFAULT",  # DEBUG | DEFAULT | WARNING | ERROR
    )

    prompt = ChatPromptTemplate.from_messages([
        ("system", "Answer concisely based on context.\nContext: {context}"),
        ("human", "{question}")
    ])
    return (prompt | llm | StrOutputParser()).invoke({
        "question": question, "context": context
    })


@observe(name="multi-turn-conversation")
def simulate_conversation(user_id: str) -> list[dict]:
    """Simulates a multi-turn conversation — all turns under one trace."""
    lf      = get_client()
    history = []

    questions = [
        "What is LangSmith?",
        "How is Langfuse different?",
        "Which one should I use for a self-hosted deployment?",
    ]

    # v4: propagate user_id/session_id for the whole conversation
    with propagate_attributes(user_id=user_id, session_id=f"conv-{user_id}"):
        for i, question in enumerate(questions):
            lf.update_current_span(metadata={"turn": i + 1, "total_turns": len(questions)})

            context = "LangSmith is LangChain's observability tool. Langfuse is open-source and self-hostable."
            answer  = generate_structured_answer(question, context, user_id)
            history.append({"turn": i+1, "q": question, "a": answer})
            print(f"Turn {i+1}: {question}")
            print(f"       {answer[:100]}...\n")

    return history

conversation = simulate_conversation("student-001")
print(f"✅ Multi-turn trace created — {len(conversation)} turns visible in one trace")


Turn 1: What is LangSmith?
       LangSmith is LangChain's observability tool....

Turn 2: How is Langfuse different?
       Langfuse is open-source and self-hostable, whereas LangSmith is a specific observability tool for La...

Turn 3: Which one should I use for a self-hosted deployment?
       You should use Langfuse for a self-hosted deployment, as it is open-source and designed for that pur...

✅ Multi-turn trace created — 3 turns visible in one trace


---
## 5. 🔧 Module 2 — Manual Tracing (Maximum Control) <a name="manual"></a>

Use the manual SDK when you need full control over the trace structure,
or when working with code you can't decorate (third-party libraries, async workers, etc.)


In [ ]:
# v4 MAJOR CHANGE: trace() / span() / generation() / event() removed from Langfuse.
# Use start_as_current_observation() + start_observation() + create_event() instead.
# span.end() now takes only end_time= — use span.update() first to set output/metadata.
# generation usage dict → usage_details= (Dict[str, int], no "unit" key)

from langfuse import Langfuse, propagate_attributes
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
import time

langfuse_client = Langfuse()
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.3)

# ── Full manual trace with spans and generations ──────────────────────────────

def process_customer_request(user_message: str, user_id: str, session_id: str) -> dict:
    """
    Fully manual trace in Langfuse v4 using start_as_current_observation().
    - span.end()  → only accepts end_time=; call span.update() first
    - generation.end() → same; use generation.update() for output/usage_details
    - usage dict → usage_details: Dict[str, int]  (no "unit" key)
    """
    final_result = {}

    # 1. Root trace via start_as_current_observation (as_type="span" is the root)
    with langfuse_client.start_as_current_observation(
        name="customer-support-request",
        as_type="span",
        input={"message": user_message},
        metadata={"channel": "web", "language": "en"},
    ) as trace_span:

        # v4: attach user_id / session_id / tags via propagate_attributes
        with propagate_attributes(
            user_id=user_id,
            session_id=session_id,
            tags=["support", "manual-tracing"],
        ):

            try:
                # ── Step 1: Intent classification ────────────────────────────
                intent_obs = trace_span.start_observation(
                    name="intent-classification",
                    as_type="span",
                    input={"text": user_message},
                    metadata={"model": "rule-based"},
                )

                time.sleep(0.05)

                intents = {
                    "refund": ["refund", "money back", "reimburse"],
                    "shipping": ["shipping", "deliver", "arrived", "late"],
                    "return": ["return", "send back", "exchange"],
                }
                detected_intent = "general"
                for intent, keywords in intents.items():
                    if any(kw in user_message.lower() for kw in keywords):
                        detected_intent = intent
                        break

                # v4: update first, then end (end only accepts end_time=)
                intent_obs.update(output={"intent": detected_intent}, level="DEFAULT")
                intent_obs.end()

                # ── Step 2: Cache-check event ─────────────────────────────────
                trace_span.create_event(
                    name="cache-check",
                    input={"query": user_message, "intent": detected_intent},
                    output={"cache_hit": False},
                    metadata={"cache_type": "semantic"},
                )

                # ── Step 3: LLM Generation ────────────────────────────────────
                prompt_template = ChatPromptTemplate.from_messages([
                    ("system", f"You are a customer support agent. Detected intent: {detected_intent}.\nBe helpful and empathetic."),
                    ("human", "{message}")
                ])

                gen_obs = trace_span.start_observation(
                    name="response-generation",
                    as_type="generation",
                    model="gpt-4o-mini",
                    model_parameters={"temperature": 0.3, "max_tokens": 300},
                    input=[
                        {"role": "system", "content": f"Support agent, intent: {detected_intent}"},
                        {"role": "user", "content": user_message},
                    ],
                    metadata={"intent": detected_intent, "prompt_version": "v3"},
                )

                gen_start    = time.time()
                chain        = prompt_template | llm | StrOutputParser()
                llm_response = chain.invoke({"message": user_message})
                gen_latency  = time.time() - gen_start

                # v4: usage_details replaces usage dict; no "unit" key; all int values
                gen_obs.update(
                    output=llm_response,
                    usage_details={
                        "input":  int(len(user_message.split()) * 1.3),
                        "output": int(len(llm_response.split()) * 1.3),
                    },
                    metadata={"latency_ms": round(gen_latency * 1000, 2)},
                )
                gen_obs.end()

                # ── Step 4: Post-processing ───────────────────────────────────
                post_obs = trace_span.start_observation(name="post-processing", as_type="span")

                response_data = {
                    "response": llm_response,
                    "intent": detected_intent,
                    "escalate": "specialist" in llm_response.lower(),
                }

                post_obs.update(output=response_data)
                post_obs.end()

                # Update root span output
                trace_span.update(
                    output=response_data,
                    metadata={
                        "total_latency_ms": round((time.time() - gen_start + 0.05) * 1000, 2),
                        "intent": detected_intent,
                        "escalated": response_data["escalate"],
                    },
                )

                final_result = response_data

            except Exception as e:
                trace_span.create_event(
                    name="error",
                    level="ERROR",
                    input={"error_type": type(e).__name__},
                    output={"message": str(e)},
                )
                trace_span.update(metadata={"error": str(e)})
                raise

    langfuse_client.flush()
    return final_result


# ── Test it ───────────────────────────────────────────────────────────────────
result = process_customer_request(
    user_message="I've been waiting 3 weeks for my order and need a refund!",
    user_id="customer-789",
    session_id="session-abc-123",
)

print("Response:", result["response"])
print("Intent:", result["intent"])
print("Escalate:", result.get("escalate"))
print()
print("✅ Full manual trace in Langfuse v4 with:")
print("   - Root span (start_as_current_observation)")
print("   - intent-classification span")
print("   - cache-check event (create_event)")
print("   - response-generation generation (usage_details, not usage)")
print("   - post-processing span")


Response: I’m really sorry to hear that your order hasn’t arrived yet. I completely understand how frustrating this must be for you. Let’s get this sorted out. 

Could you please provide me with your order number? I’ll check the status of your order and initiate the refund process for you. Thank you for your patience!
Intent: refund
Escalate: False

✅ Full manual trace in Langfuse v4 with:
   - Root span (start_as_current_observation)
   - intent-classification span
   - cache-check event (create_event)
   - response-generation generation (usage_details, not usage)
   - post-processing span


In [ ]:
# ── Using start_as_current_observation context manager ────────────────────────
# v4 change: type= parameter renamed to as_type=

from langfuse import Langfuse, observe, get_client

langfuse_client = Langfuse()

@observe(name="context-manager-demo")
def demo_context_manager(query: str) -> str:
    """Shows mixing @observe with start_as_current_observation."""
    lf = get_client()

    # v4: as_type= (not type=)
    with lf.start_as_current_observation(
        as_type="span",           # ← was type="span" in some older examples
        name="validation",
        input={"query": query},
    ) as validation_obs:
        is_valid = len(query) > 3
        validation_obs.update(output={"is_valid": is_valid})

    if not is_valid:
        return "Query too short"

    with lf.start_as_current_observation(
        as_type="generation",
        name="answer-generation",
        model="gpt-4o-mini",
        input=[{"role": "user", "content": query}],
    ) as gen_obs:
        answer = llm.invoke(query).content
        gen_obs.update(output=answer)

    return answer

result = demo_context_manager("Explain LLMOps in one sentence")
print("Result:", result)
print("\n✅ Context manager pattern — note as_type= (not type=)")


Result: LLMOps refers to the practices and methodologies for managing, deploying, and maintaining large language models (LLMs) in production environments to ensure their reliability, scalability, and performance.

✅ Context manager pattern — note as_type= (not type=)


---
## 6. 🦜 Module 3 — LangChain Integration with CallbackHandler <a name="langchain"></a>

`CallbackHandler` from `langfuse.langchain` is a **drop-in** for any existing LangChain code.
It intercepts all LangChain callbacks and sends traces to Langfuse automatically.

### Two Ways to Use CallbackHandler
1. **Per-call**: Pass it in `config={"callbacks": [handler]}` — traces only that call
2. **Global**: Set it as a default callback — traces all LangChain calls in the process


In [ ]:
# v4 change: CallbackHandler trace_context only accepts {trace_id, parent_span_id}.
# user_id / session_id / tags / metadata must be set via propagate_attributes()
# inside an enclosing @observe function.

from langfuse import observe, get_client, propagate_attributes
from langfuse.langchain import CallbackHandler
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough, RunnableLambda
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
from langchain_chroma import Chroma

llm        = ChatOpenAI(model="gpt-4o-mini", temperature=0.3)
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

# ── Method 1: Per-call callback ────────────────────────────────────────────────
# In v4, trace_context only carries {trace_id, parent_span_id}.
# Wrap the chain call inside @observe and use propagate_attributes() for user context.

@observe(name="langchain-demo-call")
def traced_chain_call(question: str, user_id: str) -> str:
    """Traces a LangChain chain call with user metadata in v4."""
    lf = get_client()

    with propagate_attributes(
        user_id=user_id,
        session_id="langchain-demo",
        tags=["langchain", "demo", "day-14"],
    ):
        lf.update_current_span(metadata={"lesson": "Module 3", "model": "gpt-4o-mini"})

        # Pass CallbackHandler with trace_context to link LangChain spans into this trace
        handler = CallbackHandler(
            trace_context={"trace_id": lf.get_current_trace_id()}
        )

        prompt = ChatPromptTemplate.from_messages([
            ("system", "You are an LLMOps expert. Be concise."),
            ("human", "{question}"),
        ])
        chain = prompt | llm | StrOutputParser()
        return chain.invoke({"question": question}, config={"callbacks": [handler]})


response = traced_chain_call(
    question="What does PagedAttention do in vLLM?",
    user_id="student-001",
)
print("Response:", response)
print()
get_client().flush()
print("✅ Trace sent to Langfuse via CallbackHandler (v4 pattern)")
print("   Every step: ChatPromptTemplate → ChatOpenAI → StrOutputParser")
print("   All visible in Langfuse trace tree")


/usr/local/lib/python3.12/dist-packages/langgraph/checkpoint/serde/encrypted.py:5: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit value (e.g., allowed_objects='messages' or allowed_objects='core') to suppress this warning.
  from langgraph.checkpoint.serde.jsonplus import JsonPlusSerializer


Response: PagedAttention in vLLM is an efficient attention mechanism designed to optimize memory usage and computational speed during the processing of large language models. It enables the model to handle longer sequences by paginating the attention computation, allowing it to process segments of the input sequentially rather than all at once. This reduces the memory footprint and improves performance, particularly for tasks involving long contexts.

✅ Trace sent to Langfuse via CallbackHandler (v4 pattern)
   Every step: ChatPromptTemplate → ChatOpenAI → StrOutputParser
   All visible in Langfuse trace tree


In [ ]:
# ── Building a RAG chain with Langfuse tracing ────────────────────────────────
# v4: No global LangChain handler — wrap calls in @observe + propagate_attributes()
# Removed: "from langchain_core.callbacks import set_handler" (never existed in LC v1)

# Build knowledge base
docs = [
    Document(page_content="LangSmith provides tracing, datasets, and evaluation for LangChain applications.", metadata={"source": "langsmith"}),
    Document(page_content="Langfuse is an open-source observability platform with self-hosting support.", metadata={"source": "langfuse"}),
    Document(page_content="MLflow tracks LLM experiments: prompts, configs, and metrics.", metadata={"source": "mlflow"}),
    Document(page_content="vLLM serves open-source LLMs with PagedAttention for high throughput.", metadata={"source": "vllm"}),
    Document(page_content="Semantic caching returns cached LLM responses for similar queries.", metadata={"source": "caching"}),
    Document(page_content="LiteLLM routes requests across 100+ LLM providers with cost tracking.", metadata={"source": "litellm"}),
    Document(page_content="Arize Phoenix is an open-source tracing tool supporting OpenTelemetry.", metadata={"source": "phoenix"}),
    Document(page_content="DeepEval and Promptfoo are frameworks for testing LLM prompt quality.", metadata={"source": "evals"}),
]

splitter    = RecursiveCharacterTextSplitter(chunk_size=200, chunk_overlap=30)
chunks      = splitter.split_documents(docs)
vectorstore = Chroma.from_documents(chunks, embedding=embeddings, collection_name="llmops-lf")
retriever   = vectorstore.as_retriever(search_kwargs={"k": 3})

rag_prompt = ChatPromptTemplate.from_messages([
    ("system", "Answer based only on the provided context. If unsure, say so.\n\nContext:\n{context}"),
    ("human", "{question}"),
])

def format_docs(docs):
    return "\n\n".join([f"[{d.metadata['source']}] {d.page_content}" for d in docs])

rag_chain = (
    {"context": retriever | RunnableLambda(format_docs), "question": RunnablePassthrough()}
    | rag_prompt
    | llm
    | StrOutputParser()
)

# ── Wrap RAG calls in @observe + propagate_attributes ─────────────────────────
@observe(name="rag-pipeline")
def run_rag(question: str, user_id: str) -> str:
    lf = get_client()
    with propagate_attributes(
        user_id=user_id,
        session_id="rag-demo",
        tags=["rag", "langchain", "langfuse"],
    ):
        lf.update_current_span(metadata={"pipeline": "RAG", "vector_store": "chroma", "top_k": 3})
        handler = CallbackHandler(
            trace_context={"trace_id": lf.get_current_trace_id()}
        )
        return rag_chain.invoke(question, config={"callbacks": [handler]})


questions = [
    "What makes Langfuse different from LangSmith?",
    "How does semantic caching work?",
    "What is vLLM used for?",
]

print("Running RAG pipeline with Langfuse tracing...\n")
for i, q in enumerate(questions):
    response = run_rag(q, user_id=f"student-{i+2:03d}")
    print(f"Q: {q}")
    print(f"A: {response[:150]}...\n")

get_client().flush()
print("✅ RAG traces in Langfuse — each shows: retrieval → prompt → LLM")


Running RAG pipeline with Langfuse tracing...

Q: What makes Langfuse different from LangSmith?
A: Langfuse is an open-source observability platform with self-hosting support, while LangSmith provides tracing, datasets, and evaluation specifically f...

Q: How does semantic caching work?
A: Semantic caching returns cached LLM responses for similar queries, allowing for quicker retrieval of information by leveraging previously processed re...

Q: What is vLLM used for?
A: vLLM is used for serving open-source LLMs (Large Language Models) with PagedAttention for high throughput....

✅ RAG traces in Langfuse — each shows: retrieval → prompt → LLM


In [ ]:
# ── LCEL chain with intent routing and Langfuse tracing ──────────────────────

from langchain_core.runnables import RunnableLambda
import json

@observe(name="intent-detector")
def detect_intent(question: str) -> str:
    intent_prompt = ChatPromptTemplate.from_messages([
        ("system", """Classify the question into ONE category:
[deployment, tracing, evaluation, cost, caching, other]
Reply with ONLY the category word."""),
        ("human", "{question}"),
    ])
    return (intent_prompt | llm | StrOutputParser()).invoke({"question": question}).strip().lower()


@observe(name="smart-router-chain")
def smart_rag(question: str, user_id: str) -> dict:
    """
    Smart RAG pipeline:
    1. Detects intent (traced via @observe)
    2. Runs RAG (traced via CallbackHandler linked to current trace)
    3. Returns structured result
    """
    lf = get_client()

    with propagate_attributes(
        user_id=user_id,
        tags=["smart-rag", "routed"],
    ):
        lf.update_current_span(metadata={"question": question[:50]})

        # Step 1: Detect intent
        intent = detect_intent(question)
        lf.update_current_span(metadata={"detected_intent": intent})

        # Step 2: RAG — link handler to current trace
        handler = CallbackHandler(
            trace_context={"trace_id": lf.get_current_trace_id()}
        )
        answer = rag_chain.invoke(question, config={"callbacks": [handler]})

    return {"intent": intent, "answer": answer, "question": question}


result = smart_rag("Which tool should I use for self-hosted LLM tracing?", "student-003")
print(f"Intent: {result['intent']}")
print(f"Answer: {result['answer'][:200]}")
print("\n✅ Full smart-router trace: detect-intent + RAG pipeline as nested spans")


Intent: tracing
Answer: You should use LangSmith for self-hosted LLM tracing, as it provides tracing, datasets, and evaluation specifically for LangChain applications.

✅ Full smart-router trace: detect-intent + RAG pipeline as nested spans


---
## 7. 📝 Module 4 — Prompt Management (Create, Version, Pull) <a name="prompts"></a>

Langfuse's prompt management lets you:
- Store prompts centrally with versioning
- Pull prompts at runtime (changes deploy without code releases)
- Link prompts to traces (know which prompt version generated which output)
- Compare performance across prompt versions


In [ ]:
from langfuse import Langfuse
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

langfuse_client = Langfuse()
llm             = ChatOpenAI(model="gpt-4o-mini", temperature=0.3)

# ─────────────────────────────────────────────────────────────────────────────
# STEP 1: Create prompt version 1
# ─────────────────────────────────────────────────────────────────────────────

prompt_v1 = langfuse_client.create_prompt(
    name="support-agent",
    prompt="""You are a helpful customer support agent.

User message: {{user_message}}""",
    # Note: Langfuse uses {{variable}} syntax (double curly braces)
    config={
        "model": "gpt-4o-mini",
        "temperature": 0.3,
        "max_tokens": 300
    },
    labels=["latest"],       # Label this as 'latest'
    tags=["support", "v1"]
)

print(f"✅ Created prompt version 1")
print(f"   Name: {prompt_v1.name}")
print(f"   Version: {prompt_v1.version}")
print(f"   Labels: {prompt_v1.labels}")


✅ Created prompt version 1
   Name: support-agent
   Version: 1
   Labels: ['latest']


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# STEP 2: Update to version 2 — improved with persona and guidelines
# ─────────────────────────────────────────────────────────────────────────────

prompt_v2 = langfuse_client.create_prompt(
    name="support-agent",   # Same name → creates a new version
    prompt="""You are a helpful, empathetic customer support agent.

Guidelines:
- Acknowledge the customer's issue before solving it
- Keep responses under 150 words
- Always offer a next step or solution
- End by asking if there's anything else you can help with

User message: {{user_message}}""",
    config={
        "model": "gpt-4o-mini",
        "temperature": 0.2,   # Lower temp for more consistent support responses
        "max_tokens": 300
    },
    labels=["latest"],        # Moves 'latest' label to this version
    tags=["support", "v2", "improved"]
)

print(f"✅ Created prompt version 2")
print(f"   Version: {prompt_v2.version}")
print(f"   Previous v1 version: {prompt_v1.version}")


✅ Created prompt version 2
   Version: 2
   Previous v1 version: 1


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# STEP 3: Create a production-tested version 3 and promote to 'production'
# ─────────────────────────────────────────────────────────────────────────────

prompt_v3 = langfuse_client.create_prompt(
    name="support-agent",
    prompt="""You are a helpful, empathetic customer support agent for ShopEasy.

Guidelines:
- Start with empathy: acknowledge the issue first
- Be concise: under 150 words
- Always give a concrete next step (not just "we'll look into it")
- Escalation phrase: "I'll connect you with a specialist" for complex issues
- Closing: always ask "Is there anything else I can help you with?"

User message: {{user_message}}""",
    config={
        "model": "gpt-4o-mini",
        "temperature": 0.2,
        "max_tokens": 300
    },
    labels=["production", "latest"],  # Both production AND latest
    tags=["support", "v3", "production-ready"]
)

print(f"✅ Created prompt version 3 — promoted to 'production'")
print(f"   Now two labels exist: 'production' → v{prompt_v3.version}, 'latest' → v{prompt_v3.version}")


✅ Created prompt version 3 — promoted to 'production'
   Now two labels exist: 'production' → v3, 'latest' → v3


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# STEP 4: Pull prompts and use them
# ─────────────────────────────────────────────────────────────────────────────

# Pull by label — production apps always pull 'production'
production_prompt = langfuse_client.get_prompt("support-agent", label="production")
print(f"Production prompt — version: {production_prompt.version}")
print(f"Content preview: {production_prompt.prompt[:80]}...")

# Pull specific version by number
v1_prompt = langfuse_client.get_prompt("support-agent", version=1)
print(f"\nHistoric v1 — version: {v1_prompt.version}")

# Pull latest (development)
latest_prompt = langfuse_client.get_prompt("support-agent")
print(f"Latest prompt — version: {latest_prompt.version}")


Production prompt — version: 3
Content preview: You are a helpful, empathetic customer support agent for ShopEasy.

Guidelines:
...

Historic v1 — version: 1
Latest prompt — version: 3


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# STEP 5: Use the prompt with LangChain and link it to the trace
# ─────────────────────────────────────────────────────────────────────────────
# v4 change: update_current_observation() → update_current_span() or update_current_generation()

from langfuse.langchain import CallbackHandler
from langfuse import get_client, observe, propagate_attributes

production_prompt = langfuse_client.get_prompt("support-agent", label="production")

@observe(name="production-support-bot")
def run_production_bot(user_message: str, user_id: str) -> str:
    lf = get_client()

    with propagate_attributes(
        user_id=user_id,
        tags=["production", f"prompt-v{production_prompt.version}"],
    ):
        lf.update_current_span(metadata={
            "prompt_name": "support-agent",
            "prompt_version": production_prompt.version,
            "prompt_label": "production",
        })

        # Build LangChain prompt from Langfuse template
        # Langfuse uses {{var}} → LangChain needs {var}
        prompt_text = production_prompt.prompt.replace("{{user_message}}", "{user_message}")

        langchain_prompt = ChatPromptTemplate.from_messages([("human", prompt_text)])
        chain = langchain_prompt | llm | StrOutputParser()

        handler = CallbackHandler(
            trace_context={"trace_id": lf.get_current_trace_id()}
        )
        response = chain.invoke({"user_message": user_message}, config={"callbacks": [handler]})

        # v4: update_current_generation() links the prompt object to the trace
        lf.update_current_generation(prompt=production_prompt)

    return response


test_messages = [
    "My order from 2 weeks ago still hasn't arrived",
    "I got the wrong item and need to exchange it",
    "I was charged twice for the same order",
]

print("Testing production bot (prompt v3):\n")
for msg in test_messages:
    response = run_production_bot(msg, user_id="customer-001")
    print(f"Q: {msg}")
    print(f"A: {response[:180]}...\n")

get_client().flush()
print("✅ Traces linked to 'support-agent' prompt v3 in Langfuse")
print("   In UI: Prompts → support-agent → see which traces used each version")


Testing production bot (prompt v3):

Q: My order from 2 weeks ago still hasn't arrived
A: I understand how frustrating it can be to wait for an order, especially when you were expecting it by now. Let’s get this sorted out for you. 

Could you please provide me with you...

Q: I got the wrong item and need to exchange it
A: I’m sorry to hear that you received the wrong item. I understand how frustrating that can be. To initiate an exchange, please provide me with your order number and the item you rec...

Q: I was charged twice for the same order
A: I’m really sorry to hear that you were charged twice for your order; I can imagine how frustrating that must be. To resolve this quickly, I’ll need to look into your order details....

✅ Traces linked to 'support-agent' prompt v3 in Langfuse
   In UI: Prompts → support-agent → see which traces used each version


---
## 8. ⭐ Module 5 — Scoring & Evaluation <a name="scoring"></a>

Langfuse scoring lets you attach quality scores to any trace or observation.
Scores can come from:
- **Automated evaluators** (keyword match, length, LLM-as-judge)
- **User feedback** (thumbs up/down, star ratings)
- **Human annotation** (expert review queue in Langfuse UI)


In [ ]:
# v4 change: update_current_observation() → update_current_span() or update_current_generation()
# score_current_trace() is unchanged and still works fine.

from langfuse import Langfuse, observe, get_client, propagate_attributes
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
import re, json

langfuse_client = Langfuse()
llm             = ChatOpenAI(model="gpt-4o-mini", temperature=0.3)

@observe(name="scored-pipeline")
def scored_pipeline(user_message: str) -> dict:
    """Pipeline that self-scores its own output after generation."""
    lf = get_client()

    prompt = ChatPromptTemplate.from_messages([
        ("system", """You are a helpful support agent.
Acknowledge the issue, offer a concrete solution, end with "Is there anything else I can help with?"""),
        ("human", "{message}"),
    ])
    chain    = prompt | llm | StrOutputParser()
    response = chain.invoke({"message": user_message})

    # ── Automated scoring ─────────────────────────────────────────────────────
    has_closing = "is there anything else" in response.lower()
    lf.score_current_trace(
        name="has_closing_phrase",
        value=1.0 if has_closing else 0.0,
        comment="Closing phrase check",
    )

    word_count = len(response.split())
    if 50 <= word_count <= 200:
        length_score = 1.0
    elif word_count < 50:
        length_score = word_count / 50
    else:
        length_score = max(0, 1 - (word_count - 200) / 200)

    lf.score_current_trace(
        name="response_length_score",
        value=length_score,
        comment=f"{word_count} words",
    )

    empathy_words = ["sorry", "understand", "apologize", "frustrating"]
    empathy_score = min(1.0, sum(1 for w in empathy_words if w in response.lower()) / 2)
    lf.score_current_trace(
        name="empathy_score",
        value=empathy_score,
        comment="Empathy word check",
    )

    return {
        "response": response,
        "scores": {
            "closing_phrase": float(has_closing),
            "length": round(length_score, 2),
            "empathy": round(empathy_score, 2),
        },
    }


result = scored_pipeline("I've been waiting 3 weeks and my package still hasn't arrived!")
print("Response:", result["response"])
print("\nScores attached to trace:")
for key, val in result["scores"].items():
    bar = "█" * int(val * 20) + "░" * (20 - int(val * 20))
    print(f"  {key:<25} {bar} {val:.0%}")

get_client().flush()


Response: I understand how frustrating it can be to wait for a package for such a long time. To help resolve this issue, I recommend checking the tracking information for your package, if you have it. This can provide updates on its current status. If the tracking shows no movement or if it seems lost, please contact the shipping carrier directly or reach out to the retailer from whom you ordered the item for further assistance. They may be able to initiate a trace or provide a refund or replacement. 

Is there anything else I can help with?

Scores attached to trace:
  closing_phrase            ████████████████████ 100%
  length                    ████████████████████ 100%
  empathy                   ████████████████████ 100%


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# STEP 2: Generate traces to score
# v4 change: start_as_current_observation(type=...) → as_type=...
#            "trace" is not a valid as_type; use "span" for root observations
# ─────────────────────────────────────────────────────────────────────────────

trace_ids = []

test_inputs = [
    "My order is 2 weeks late",
    "I received a broken product",
    "I want to cancel my subscription",
    "I was overcharged on my last order",
    "How do I track my shipment?",
]

print("Generating traces to score...\n")
for msg in test_inputs:

    @observe(name="eval-target")
    def generate_response(message):
        prompt = ChatPromptTemplate.from_messages([
            ("system", "You are a helpful support agent. Be concise and empathetic."),
            ("human", "{message}"),
        ])
        return (prompt | llm | StrOutputParser()).invoke({"message": message})

    # v4: as_type="span" (not type="trace") — root span acts as the trace root
    with get_client().start_as_current_observation(
        as_type="span",                         # ← was type="trace" (invalid in v4)
        name="support-response-for-eval",
        input={"message": msg},
    ) as obs:
        response = generate_response(msg)
        obs.update(output={"response": response})
        trace_ids.append(get_client().get_current_trace_id())
        print(f"  ✓ Generated response for: {msg[:40]}...")

get_client().flush()
print(f"\n✅ {len(trace_ids)} traces ready for evaluation")


Generating traces to score...

  ✓ Generated response for: My order is 2 weeks late...
  ✓ Generated response for: I received a broken product...
  ✓ Generated response for: I want to cancel my subscription...
  ✓ Generated response for: I was overcharged on my last order...
  ✓ Generated response for: How do I track my shipment?...

✅ 5 traces ready for evaluation


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# STEP 3: Score traces externally (post-hoc LLM-as-judge evaluation)
# create_score() API is unchanged in v4.
# ─────────────────────────────────────────────────────────────────────────────

def llm_judge_score(response_text: str, customer_message: str) -> dict:
    """LLM-as-judge scoring using gpt-4o-mini."""
    judge_prompt = ChatPromptTemplate.from_messages([
        ("system", """You score customer support responses.
Rate helpfulness from 1-5:
1 = Unhelpful/harmful  3 = Adequate  5 = Excellent

Reply ONLY as JSON: {"score": <1-5>, "reason": "<one sentence>"}
No other text."""),
        ("human", "Customer: {customer}\nAgent: {agent}\nRate:"),
    ])

    judge_llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

    try:
        result    = judge_llm.invoke(
            judge_prompt.format_messages(customer=customer_message, agent=response_text)
        )
        json_text = re.search(r'\{.*?\}', result.content, re.DOTALL)
        if json_text:
            data  = json.loads(json_text.group())
            return {"score": data["score"] / 5.0, "reason": data.get("reason", "")}
    except Exception:
        pass
    return {"score": 0.5, "reason": "Evaluation failed"}


# Score each trace
print("Scoring traces with LLM judge...\n")
for i, (trace_id, msg) in enumerate(zip(trace_ids, test_inputs)):
    prompt    = ChatPromptTemplate.from_messages([
        ("system", "You are a helpful support agent. Be concise and empathetic."),
        ("human", "{message}"),
    ])
    response  = (prompt | llm | StrOutputParser()).invoke({"message": msg})

    judge_result = llm_judge_score(response, msg)

    langfuse_client.create_score(
        trace_id=trace_id,
        name="llm_helpfulness",
        value=judge_result["score"],
        comment=judge_result["reason"],
        data_type="NUMERIC",
    )

    print(f"  Trace {i+1}: score={judge_result['score']:.2f} | {judge_result['reason'][:60]}...")

langfuse_client.flush()
print("\n✅ All traces scored — visible in Langfuse under Traces → Scores filter")


Scoring traces with LLM judge...

  Trace 1: score=0.50 | Evaluation failed...
  Trace 2: score=0.50 | Evaluation failed...
  Trace 3: score=0.50 | Evaluation failed...
  Trace 4: score=0.50 | Evaluation failed...
  Trace 5: score=0.50 | Evaluation failed...

✅ All traces scored — visible in Langfuse under Traces → Scores filter


---
## 9. 🧪 Module 6 — Datasets & Experiments <a name="datasets"></a>

Langfuse datasets work similarly to LangSmith datasets but with a different API.
Use them to run systematic experiments and compare prompt/model versions.


In [ ]:
from langfuse import Langfuse
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

langfuse_client = Langfuse()
llm             = ChatOpenAI(model="gpt-4o-mini", temperature=0.3)

# ─────────────────────────────────────────────────────────────────────────────
# STEP 1: Create a dataset with examples
# ─────────────────────────────────────────────────────────────────────────────

DATASET_NAME = "support-bot-golden-lf"

try:
    dataset = langfuse_client.create_dataset(
        name=DATASET_NAME,
        description="Golden dataset for customer support bot evaluation in Langfuse",
        metadata={"version": "1.0", "created_by": "instructor", "module": "day-14"}
    )
    print(f"✅ Dataset created: {DATASET_NAME}")
except Exception as e:
    print(f"Dataset may already exist: {e}")
    dataset = langfuse_client.get_dataset(DATASET_NAME)
    print(f"✅ Using existing dataset: {DATASET_NAME}")

# Define golden examples
golden_examples = [
    {
        "input":            {"user_message": "My order hasn't arrived in 3 weeks"},
        "expected_output":  {"must_contain": ["order", "sorry"], "intent": "shipping"}
    },
    {
        "input":            {"user_message": "I received a damaged product and want a refund"},
        "expected_output":  {"must_contain": ["sorry", "refund"], "intent": "refund"}
    },
    {
        "input":            {"user_message": "I want to return an item I bought last month"},
        "expected_output":  {"must_contain": ["return"], "intent": "return"}
    },
    {
        "input":            {"user_message": "I was charged twice for my order"},
        "expected_output":  {"must_contain": ["sorry", "charge"], "intent": "billing"}
    },
    {
        "input":            {"user_message": "How do I track my shipment?"},
        "expected_output":  {"must_contain": ["track"], "intent": "info"}
    },
    {
        "input":            {"user_message": "I want to cancel my subscription immediately"},
        "expected_output":  {"must_contain": ["cancel", "subscription"], "intent": "cancellation"}
    },
]

# Upload examples to dataset
for ex in golden_examples:
    langfuse_client.create_dataset_item(
        dataset_name=DATASET_NAME,
        input=ex["input"],
        expected_output=ex["expected_output"],
    )

print(f"✅ Uploaded {len(golden_examples)} examples to dataset")


✅ Dataset created: support-bot-golden-lf
✅ Uploaded 6 examples to dataset


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# STEP 2: Run an experiment — test prompt v1 against dataset
# v4 MAJOR CHANGE: item.run() context manager removed.
# Use dataset.run_experiment(task=..., evaluators=[...]) instead.
# ─────────────────────────────────────────────────────────────────────────────

def run_experiment(prompt_text: str, experiment_name: str, run_description: str = ""):
    """
    v4 pattern: dataset.run_experiment() with a task function.
    Langfuse automatically links each task execution to the dataset item
    and groups all runs under the experiment name.
    """
    dataset = langfuse_client.get_dataset(DATASET_NAME)

    print(f"\n🚀 Running experiment: {experiment_name}")

    langchain_prompt = ChatPromptTemplate.from_messages([
        ("system", prompt_text),
        ("human", "{user_message}"),
    ])
    chain = langchain_prompt | llm | StrOutputParser()

    # ── task function: called once per dataset item ──────────────────────────
    def task(*, item, **kwargs):
        user_message = item.input.get("user_message", "")
        return chain.invoke({"user_message": user_message})

    # ── evaluator: called with (input, output, expected_output, ...) ─────────
    def keyword_evaluator(*, input, output, expected_output=None, **kwargs):
        expected = (expected_output or {}).get("must_contain", [])
        if not expected:
            return {"name": "keyword_coverage", "value": 1.0, "comment": "No keywords to check"}
        found = sum(1 for kw in expected if kw.lower() in output.lower())
        score = found / len(expected)
        return {
            "name": "keyword_coverage",
            "value": score,
            "comment": f"Found {found}/{len(expected)} required keywords",
        }

    result = dataset.run_experiment(
        name=experiment_name,
        description=run_description,
        task=task,
        evaluators=[keyword_evaluator],
        max_concurrency=5,
    )

    # Print per-item results
    item_scores = []
    for item_result in result.item_results:
        score_val = item_result.evaluations[0].value if item_result.evaluations else 0
        item_scores.append(score_val)
        inp = item_result.item.input.get("user_message", "")
        print(f"  Item: score={score_val:.2f} | Q: {inp[:40]}...")

    avg_score = sum(item_scores) / len(item_scores) if item_scores else 0
    print(f"  Average score: {avg_score:.2%}")
    print(f"  View results: {result.dataset_run_url}")
    return avg_score


PROMPT_V1 = "You are a helpful customer support agent."
avg_v1 = run_experiment(PROMPT_V1, "experiment-v1", "Basic prompt, no guidelines")
langfuse_client.flush()



🚀 Running experiment: experiment-v1


ERROR:langfuse:Evaluator failed: 'dict' object has no attribute 'name'
ERROR:langfuse:Evaluator failed: 'dict' object has no attribute 'name'
ERROR:langfuse:Evaluator failed: 'dict' object has no attribute 'name'
ERROR:langfuse:Evaluator failed: 'dict' object has no attribute 'name'
ERROR:langfuse:Evaluator failed: 'dict' object has no attribute 'name'
ERROR:langfuse:Evaluator failed: 'dict' object has no attribute 'name'


AttributeError: 'dict' object has no attribute 'value'

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# STEP 3: Run experiment with improved prompt v3 and compare
# ─────────────────────────────────────────────────────────────────────────────

PROMPT_V3 = """You are a helpful, empathetic customer support agent.
- Acknowledge the issue with empathy
- Provide a concrete solution or next step
- Keep response under 150 words
- End by asking if there's anything else you can help with"""

avg_v3 = run_experiment(PROMPT_V3, "experiment-v3", "Improved prompt with guidelines")
langfuse_client.flush()

print(f"\n=== Experiment Results ===")
print(f"  Prompt v1 average keyword score: {avg_v1:.2%}")
print(f"  Prompt v3 average keyword score: {avg_v3:.2%}")
print(f"  Improvement: {(avg_v3 - avg_v1):+.2%}")
print()
print("✅ Both experiments visible in Langfuse:")
print("   Datasets → support-bot-golden-lf → Run Comparison")


---
## 10. 🗄️ Module 7 — Full Production RAG Pipeline <a name="rag"></a>
Bringing everything together: `@observe` + `CallbackHandler` + `Langfuse prompts` + scoring.


In [ ]:
from langfuse import Langfuse, observe, get_client
from langfuse.langchain import CallbackHandler
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough, RunnableLambda
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
from langchain_chroma import Chroma
import time

langfuse_client = Langfuse()
llm             = ChatOpenAI(model="gpt-4o-mini", temperature=0.3)
embeddings      = OpenAIEmbeddings(model="text-embedding-3-small")

# ── Create and store RAG prompt in Langfuse ───────────────────────────────────
rag_prompt_lf = langfuse_client.create_prompt(
    name="llmops-rag-prompt",
    prompt="""You are an LLMOps expert assistant. Answer the question based strictly on the context.
If the context doesn't contain the answer, say "I don't have information on that topic."

Context:
{{context}}

Question: {{question}}""",
    config={"model": "gpt-4o-mini", "temperature": 0.3},
    labels=["production", "latest"],
    tags=["rag", "llmops"]
)

print(f"✅ RAG prompt created in Langfuse (version {rag_prompt_lf.version})")

# ── Build knowledge base ──────────────────────────────────────────────────────
DOCS = [
    Document(page_content="LangSmith provides distributed tracing, dataset management, and evaluation for LangChain apps. It integrates automatically via LANGCHAIN_TRACING_V2 environment variable.", metadata={"source": "langsmith", "topic": "tracing"}),
    Document(page_content="Langfuse is open-source, self-hostable, and works with any LLM framework. It provides tracing via @observe decorator or CallbackHandler, prompt versioning, datasets, and scoring.", metadata={"source": "langfuse", "topic": "tracing"}),
    Document(page_content="MLflow tracks LLM experiments with parameters, metrics, and artifacts. The model registry stores versioned models and allows promoting to production status.", metadata={"source": "mlflow", "topic": "versioning"}),
    Document(page_content="vLLM uses PagedAttention to manage KV cache memory efficiently, enabling continuous batching and 2-10x higher throughput than naive LLM serving.", metadata={"source": "vllm", "topic": "deployment"}),
    Document(page_content="Semantic caching stores LLM responses indexed by semantic meaning using vector similarity. A similarity threshold (0.90-0.95) determines when to serve the cached response.", metadata={"source": "caching", "topic": "optimization"}),
    Document(page_content="LiteLLM provides a unified interface to 100+ LLM providers with model routing, fallbacks, load balancing, and per-request cost tracking.", metadata={"source": "litellm", "topic": "cost"}),
    Document(page_content="Arize Phoenix is open-source, uses OpenTelemetry, and provides trace visualization, embedding clustering, and eval integration without requiring an account.", metadata={"source": "phoenix", "topic": "tracing"}),
    Document(page_content="Promptfoo is a CLI/SDK for testing LLM prompts with assertions. It integrates with GitHub Actions to run prompt evals on every pull request.", metadata={"source": "promptfoo", "topic": "ci-cd"}),
]

splitter    = RecursiveCharacterTextSplitter(chunk_size=250, chunk_overlap=40)
chunks      = splitter.split_documents(DOCS)
vectorstore = Chroma.from_documents(chunks, embedding=embeddings, collection_name="llmops-prod")
retriever   = vectorstore.as_retriever(search_kwargs={"k": 3})

print(f"✅ Knowledge base: {len(chunks)} chunks from {len(DOCS)} documents")


✅ RAG prompt created in Langfuse (version 1)
✅ Knowledge base: 8 chunks from 8 documents


In [ ]:
# ── Full production RAG pipeline with all Langfuse v4 features ────────────────

def format_docs(docs) -> str:
    return "\n\n---\n\n".join([
        f"[Source: {d.metadata.get('source', '?')}]\n{d.page_content}"
        for d in docs
    ])


@observe(name="production-rag-pipeline")
def production_rag(
    user_query:  str,
    user_id:     str,
    session_id:  str,
) -> dict:
    """
    Full production RAG pipeline — v4 pattern:
    - propagate_attributes() for user_id / session_id / tags
    - update_current_span() for metadata
    - CallbackHandler linked via trace_context={trace_id=...}
    - score_current_trace() for automated quality scores
    """
    lf = get_client()

    with propagate_attributes(
        user_id=user_id,
        session_id=session_id,
        tags=["production", "rag", "llmops-kb"],
    ):
        lf.update_current_span(metadata={
            "query_length": str(len(user_query.split())),
            "pipeline_version": "v2.1",
        })

        # Pull latest production prompt from Langfuse
        prompt_obj  = langfuse_client.get_prompt("llmops-rag-prompt", label="production")
        prompt_text = (
            prompt_obj.prompt
            .replace("{{context}}", "{context}")
            .replace("{{question}}", "{question}")
        )

        lc_prompt = ChatPromptTemplate.from_messages([("human", prompt_text)])

        # Link LangChain spans into this trace
        lf_handler = CallbackHandler(
            trace_context={"trace_id": lf.get_current_trace_id()}
        )

        rag_chain = (
            {"context": retriever | RunnableLambda(format_docs), "question": RunnablePassthrough()}
            | lc_prompt
            | llm
            | StrOutputParser()
        )

        t0       = time.time()
        response = rag_chain.invoke(user_query, config={"callbacks": [lf_handler]})
        latency  = round((time.time() - t0) * 1000, 2)

        # Automated quality scores
        word_count   = len(response.split())
        length_score = 1.0 if 50 <= word_count <= 300 else max(0, 1 - abs(word_count - 175) / 175)
        lf.score_current_trace(name="response_length", value=length_score, comment=f"{word_count} words")

        has_uncertainty = "i don't have information" in response.lower()
        lf.score_current_trace(
            name="appropriate_uncertainty",
            value=1.0,
            comment="Admitted knowledge gap" if has_uncertainty else "Answered from context",
        )

        lf.update_current_span(metadata={
            "latency_ms": str(latency),
            "word_count": str(word_count),
            "prompt_version": str(prompt_obj.version),
        })

    return {
        "query":          user_query,
        "response":       response,
        "latency_ms":     latency,
        "prompt_version": prompt_obj.version,
    }


test_queries = [
    ("What is the difference between Langfuse and LangSmith?", "user-001"),
    ("How does vLLM improve LLM inference throughput?",         "user-002"),
    ("What is semantic caching and when should I use it?",      "user-003"),
    ("How does Promptfoo integrate with GitHub Actions?",        "user-004"),
]

print("Running production RAG pipeline...\n")
for query, uid in test_queries:
    result = production_rag(user_query=query, user_id=uid, session_id=f"session-{uid}")
    print(f"Q: {query}")
    print(f"A: {result['response'][:200]}...")
    print(f"   Latency: {result['latency_ms']}ms | Prompt v{result['prompt_version']}\n")

langfuse_client.flush()
print("✅ All production traces in Langfuse with scores, user IDs, and session grouping")


Running production RAG pipeline...

Q: What is the difference between Langfuse and LangSmith?
A: LangSmith provides distributed tracing, dataset management, and evaluation specifically for LangChain apps, integrating automatically via the LANGCHAIN_TRACING_V2 environment variable. In contrast, La...
   Latency: 2217.92ms | Prompt v1

Q: How does vLLM improve LLM inference throughput?
A: vLLM improves LLM inference throughput by using PagedAttention to manage KV cache memory efficiently, enabling continuous batching and achieving 2-10x higher throughput than naive LLM serving....
   Latency: 1745.8ms | Prompt v1

Q: What is semantic caching and when should I use it?
A: Semantic caching stores LLM responses indexed by semantic meaning using vector similarity. You should use it when you want to serve cached responses based on a similarity threshold (0.90-0.95), which ...
   Latency: 1402.87ms | Prompt v1

Q: How does Promptfoo integrate with GitHub Actions?
A: Promptfoo integrates with Gi

---
## 11. 🔄 Module 8 — Langfuse vs LangSmith Implementation Comparison <a name="comparison"></a>

Here's the same task implemented in both platforms side-by-side.


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# SIDE-BY-SIDE COMPARISON
# Same task: "trace a LangChain chain and attach metadata"
# ══════════════════════════════════════════════════════════════════════════════

print("""
╔══════════════════════════════════════════════════════════════════════╗
║              LangSmith vs Langfuse — Code Comparison                 ║
╠══════════════════════════════════════════════════════════════════════╣
║                                                                      ║
║  TASK: Trace a LangChain chain + attach user metadata               ║
║                                                                      ║
╠══════════════════════════════════════════════════════════════════════╣
║ LANGSMITH                        ║ LANGFUSE                         ║
╠══════════════════════════════════════════════════════════════════════╣
║ # Setup                          ║ # Setup                          ║
║ import os                        ║ from langfuse import Langfuse    ║
║ os.environ[                      ║ from langfuse.langchain import   ║
║   "LANGCHAIN_TRACING_V2"]="true" ║   CallbackHandler                ║
║ os.environ[                      ║                                  ║
║   "LANGCHAIN_API_KEY"] = key     ║ lf = Langfuse(                   ║
║                                  ║   public_key=pk,                 ║
║ # Chain runs auto-traced!        ║   secret_key=sk)                 ║
║ chain.invoke(inputs)             ║                                  ║
║                                  ║ handler = CallbackHandler(       ║
║ # With metadata:                 ║   trace_context={                ║
║ chain.invoke(inputs, config={    ║     "user_id": "u-1",           ║
║   "run_name": "my-run",          ║     "tags": ["prod"]            ║
║   "tags": ["prod"],              ║   })                             ║
║   "metadata": {"uid": "u-1"}     ║                                  ║
║ })                               ║ # Pass handler:                  ║
║                                  ║ chain.invoke(inputs, config={    ║
║                                  ║   "callbacks": [handler]})       ║
╠══════════════════════════════════════════════════════════════════════╣
║ PROMPT VERSIONING                                                    ║
╠══════════════════════════════════════════════════════════════════════╣
║ # Push                           ║ # Create                         ║
║ client.push_prompt(              ║ lf.create_prompt(                ║
║   "my-prompt",                   ║   name="my-prompt",              ║
║   object=lc_prompt,              ║   prompt="Answer: {{q}}",        ║
║   tags=["v1"]                    ║   labels=["production"])         ║
║ )                                ║                                  ║
║                                  ║ # Pull                           ║
║ # Pull                           ║ p = lf.get_prompt(               ║
║ p = client.pull_prompt(          ║   "my-prompt",                   ║
║   "my-prompt:latest")            ║   label="production")            ║
╠══════════════════════════════════════════════════════════════════════╣
║ SCORING / EVALUATION                                                 ║
╠══════════════════════════════════════════════════════════════════════╣
║ client.create_feedback(          ║ lf.create_score(                 ║
║   run_id=run_id,                 ║   trace_id=trace_id,             ║
║   key="quality",                 ║   name="quality",                ║
║   score=0.9)                     ║   value=0.9)                     ║
║                                  ║                                  ║
║ # Or inside @traceable:          ║ # Or inside @observe:            ║
║ # (no direct method —            ║ get_client()                     ║
║ #  use run_helpers.get_current   ║   .score_current_trace(          ║
║ #  _run_tree)                    ║     name="q", value=0.9)         ║
╚══════════════════════════════════════════════════════════════════════╝
""")



╔══════════════════════════════════════════════════════════════════════╗
║              LangSmith vs Langfuse — Code Comparison                 ║
╠══════════════════════════════════════════════════════════════════════╣
║                                                                      ║
║  TASK: Trace a LangChain chain + attach user metadata               ║
║                                                                      ║
╠══════════════════════════════════════════════════════════════════════╣
║ LANGSMITH                        ║ LANGFUSE                         ║
╠══════════════════════════════════════════════════════════════════════╣
║ # Setup                          ║ # Setup                          ║
║ import os                        ║ from langfuse import Langfuse    ║
║ os.environ[                      ║ from langfuse.langchain import   ║
║   "LANGCHAIN_TRACING_V2"]="true" ║   CallbackHandler                ║
║ os.environ[                      ║                    

---
## 🎓 Summary — What You've Built with Langfuse

| Feature | What You Did |
|---------|-------------|
| **@observe** | Decorated Python functions create traces automatically |
| **as_type="generation"** | LLM calls tagged as generation spans with token tracking |
| **get_client()** | Updated trace metadata, user IDs, session IDs from inside functions |
| **Manual tracing** | Full control: trace → span → generation → event → end() |
| **Context manager** | `start_as_current_observation()` for fine-grained control |
| **CallbackHandler** | Drop-in LangChain integration with trace context |
| **RAG chain tracing** | Full LCEL chain traced: retrieval → prompt → LLM → parser |
| **Prompt v1/v2/v3** | Versioned prompts with labels: `latest`, `production` |
| **Prompt pull** | `get_prompt(label="production")` loads latest promoted version |
| **Prompt-trace linking** | Traces know which prompt version generated them |
| **score_current_trace** | Automated scores attached: length, empathy, keyword coverage |
| **LLM judge** | GPT-4o-mini scores runs post-hoc via `create_score()` |
| **Datasets** | Golden examples uploaded with `create_dataset_item()` |
| **Experiments** | `item.run()` groups traces into comparable experiment runs |
| **Full Production RAG** | @observe + CallbackHandler + versioned prompt + scoring in one pipeline |

---
### 🔁 Langfuse Workflow Summary

```
1. DEVELOP  →  Add @observe or CallbackHandler to your code
2. ITERATE  →  Create/update prompts in Langfuse, pull via get_prompt()
3. TEST     →  Build golden datasets, run experiments, compare scores  
4. DEPLOY   →  Set prompt label to "production", apps auto-pull updated prompts
5. MONITOR  →  Scores, user analytics, session traces, cost tracking in UI
```

> **Key advantage over LangSmith:** Langfuse is fully open-source and self-hostable.  
> For HIPAA/GDPR sensitive data, deploy Langfuse with `docker-compose up` on your own infra.
